# YOLOv8 Training on SKU-110K Dataset

**Project**: Retail Shelf Monitoring - Challenge 1: Out-of-Stock Detection  
**Task**: T028 - Train YOLOv8s model on SKU-110K (50 epochs, single "object" class)  
**Date**: December 27, 2025

---

## Overview

This notebook trains a YOLOv8 model to detect products in retail shelf images. The model learns to identify products as a single "object" class, which will then be used for:
1. **Gap Detection** - Identifying empty shelf spaces (out-of-stock areas)
2. **Object Counting** - Counting total products on shelves

### Dataset: SKU-110K
- **Images**: 11,743 retail shelf images
- **Split**: 70% train, 15% validation, 15% test
- **Class**: Single "object" class (generic product detection)
- **Format**: YOLO format (normalized bounding boxes)

### Training Configuration
- **Model**: YOLOv8s (small model, good balance of speed vs accuracy)
- **Epochs**: 50 (with early stopping patience=10)
- **Batch Size**: 16
- **Image Size**: 640x640
- **Optimizer**: AdamW
- **Learning Rate**: 0.01 → 0.0001 (cosine schedule)

### Expected Performance
- **Precision**: >90%
- **Recall**: >85%
- **mAP@0.5**: >85%
- **Inference Speed**: ~300-400ms on M1/M2 Mac

## 1. Setup and Imports

First, let's import all necessary libraries and set up our environment.

In [ ]:
import os
import sys
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
import numpy as np
from datetime import datetime

# Add project root to path so we can import our modules
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import Ultralytics YOLO
from ultralytics import YOLO

# Configure matplotlib
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

# Create output directory for plots
output_dir = Path("../outputs/training")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"✅ Setup complete")
print(f"📁 Project root: {project_root}")
print(f"📊 Output directory: {output_dir}")

## 2. Dataset Validation

Let's verify that the SKU-110K dataset is properly formatted and ready for training.

In [ ]:
# Load dataset configuration
data_yaml_path = project_root / "data" / "processed" / "SKU110K_yolo" / "data.yaml"

if not data_yaml_path.exists():
    print(f"❌ Error: Dataset configuration not found at {data_yaml_path}")
    print("   Please run: python scripts/prepare_data.py")
else:
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    print("✅ Dataset configuration loaded")
    print(f"\n📋 Configuration:")
    print(f"   Path: {data_config['path']}")
    print(f"   Classes: {data_config['nc']} ({data_config['names']})")
    print(f"   Train: {data_config['train']}")
    print(f"   Val: {data_config['val']}")
    print(f"   Test: {data_config['test']}")
    
    # Count images in each split
    dataset_root = Path(data_config['path'])
    for split in ['train', 'val', 'test']:
        split_dir = dataset_root / data_config[split] / 'images'
        if split_dir.exists():
            image_count = len(list(split_dir.glob('*.jpg')))
            print(f"\n   {split.capitalize()} images: {image_count:,}")
        else:
            print(f"   ❌ {split.capitalize()} directory not found!")
    
    # Check label files
    train_labels = dataset_root / data_config['train'] / 'labels'
    if train_labels.exists():
        label_count = len(list(train_labels.glob('*.txt')))
        print(f"\n   Train labels: {label_count:,}")
    
    print(f"\n✅ Dataset validation complete")

## 3. Visualize Sample Images

Let's look at some training images with their annotations to understand the data.

In [ ]:
def visualize_yolo_annotations(image_path, label_path, max_boxes=50):
    """
    Visualize YOLO format annotations on image.
    YOLO format: class_id x_center y_center width height (all normalized 0-1)
    """
    # Read image
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # Read labels
    if label_path.exists():
        with open(label_path, 'r') as f:
            lines = f.readlines()
        
        # Draw bounding boxes (limit to max_boxes for readability)
        for i, line in enumerate(lines[:max_boxes]):
            class_id, x_center, y_center, width, height = map(float, line.strip().split())
            
            # Convert from YOLO format to pixel coordinates
            x1 = int((x_center - width/2) * w)
            y1 = int((y_center - height/2) * h)
            x2 = int((x_center + width/2) * w)
            y2 = int((y_center + height/2) * h)
            
            # Draw rectangle
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        box_count = len(lines)
        displayed = min(box_count, max_boxes)
    else:
        box_count = 0
        displayed = 0
    
    return img, box_count, displayed

# Visualize 4 random training images
train_images_dir = dataset_root / data_config['train'] / 'images'
train_labels_dir = dataset_root / data_config['train'] / 'labels'
train_images = list(train_images_dir.glob('*.jpg'))

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

np.random.seed(42)
sample_indices = np.random.choice(len(train_images), 4, replace=False)

for idx, ax in enumerate(axes):
    img_path = train_images[sample_indices[idx]]
    label_path = train_labels_dir / f"{img_path.stem}.txt"
    
    img, total_boxes, displayed_boxes = visualize_yolo_annotations(img_path, label_path)
    
    ax.imshow(img)
    ax.set_title(f"{img_path.name}\n{displayed_boxes}/{total_boxes} products (green boxes)")
    ax.axis('off')

plt.tight_layout()
plt.savefig(output_dir / 'sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Sample images saved to {output_dir / 'sample_images.png'}")

## 4. Configure Training

Set up the YOLOv8 model and training hyperparameters.

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    'model': 'yolov8s.pt',  # Small model (11M params, 21.5MB)
    'epochs': 50,
    'batch': 16,
    'imgsz': 640,
    'patience': 10,  # Early stopping
    'device': '',  # Auto-detect GPU/CPU
    'optimizer': 'AdamW',
    'lr0': 0.01,  # Initial learning rate
    'lrf': 0.01,  # Final learning rate (lr0 * lrf)
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'save_period': 10,  # Save checkpoint every 10 epochs
    'workers': 8,
    'project': str(project_root / 'runs' / 'detect'),
    'name': 'train',
    'exist_ok': True
}

print("📋 Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"   {key}: {value}")

# Initialize model
print(f"\n🔄 Loading model: {TRAINING_CONFIG['model']}")
model = YOLO(TRAINING_CONFIG['model'])

print(f"✅ Model loaded successfully")
print(f"   Architecture: YOLOv8s")
print(f"   Parameters: ~11M")
print(f"   Pretrained: ImageNet + COCO")

## 5. Start Training

**⚠️ Training Time**: This will take 2-3 hours on M1/M2 Mac CPU (~3-4 minutes per epoch × 50 epochs)

The training will:
1. Load the pretrained YOLOv8s model
2. Fine-tune on SKU-110K dataset (8,219 train images)
3. Validate after each epoch (588 val images)
4. Save checkpoints every 10 epochs
5. Stop early if no improvement for 10 epochs
6. Save the best model based on mAP@0.5

**Note**: You can stop training with Kernel → Interrupt and resume later using the `resume=True` parameter.

In [ ]:
# Start training
start_time = datetime.now()
print(f"🚀 Training started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📊 Progress will be displayed below...\n")

try:
    results = model.train(
        data=str(data_yaml_path),
        epochs=TRAINING_CONFIG['epochs'],
        batch=TRAINING_CONFIG['batch'],
        imgsz=TRAINING_CONFIG['imgsz'],
        device=TRAINING_CONFIG['device'],
        patience=TRAINING_CONFIG['patience'],
        optimizer=TRAINING_CONFIG['optimizer'],
        lr0=TRAINING_CONFIG['lr0'],
        lrf=TRAINING_CONFIG['lrf'],
        momentum=TRAINING_CONFIG['momentum'],
        weight_decay=TRAINING_CONFIG['weight_decay'],
        warmup_epochs=TRAINING_CONFIG['warmup_epochs'],
        save=True,
        save_period=TRAINING_CONFIG['save_period'],
        val=True,
        plots=True,
        project=TRAINING_CONFIG['project'],
        name=TRAINING_CONFIG['name'],
        exist_ok=TRAINING_CONFIG['exist_ok']
    )
    
    end_time = datetime.now()
    duration = end_time - start_time
    
    print(f"\n✅ Training completed successfully!")
    print(f"⏱️  Duration: {duration}")
    print(f"📁 Results saved to: {results.save_dir}")
    
except KeyboardInterrupt:
    print(f"\n⚠️  Training interrupted by user")
    print(f"   You can resume training later by setting resume=True")
    
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    raise

## 6. Analyze Training Results

Load and visualize training metrics from the results CSV file.

In [ ]:
# Find the latest training run
runs_dir = project_root / 'runs' / 'detect'
train_runs = sorted([d for d in runs_dir.iterdir() if d.is_dir() and d.name.startswith('train')])

if not train_runs:
    print("❌ No training runs found. Please run the training cell first.")
else:
    latest_run = train_runs[-1]
    results_csv = latest_run / 'results.csv'
    
    if not results_csv.exists():
        print(f"❌ Results file not found: {results_csv}")
    else:
        # Load training results
        df = pd.read_csv(results_csv)
        
        # Clean column names (remove leading/trailing spaces)
        df.columns = df.columns.str.strip()
        
        print(f"✅ Loaded results from: {latest_run.name}")
        print(f"   Total epochs trained: {len(df)}")
        print(f"\n📊 Final Metrics (Epoch {len(df)}):")
        
        # Get final epoch metrics
        final = df.iloc[-1]
        print(f"   Train Loss (Box): {final['train/box_loss']:.4f}")
        print(f"   Train Loss (Class): {final['train/cls_loss']:.4f}")
        print(f"   Train Loss (DFL): {final['train/dfl_loss']:.4f}")
        print(f"   Validation Precision: {final['metrics/precision(B)']:.3f}")
        print(f"   Validation Recall: {final['metrics/recall(B)']:.3f}")
        print(f"   Validation mAP@0.5: {final['metrics/mAP50(B)']:.3f}")
        print(f"   Validation mAP@0.5:0.95: {final['metrics/mAP50-95(B)']:.3f}")
        
        print(f"\n🏆 Best Metrics:")
        print(f"   Best mAP@0.5: {df['metrics/mAP50(B)'].max():.3f} (Epoch {df['metrics/mAP50(B)'].idxmax() + 1})")
        print(f"   Best mAP@0.5:0.95: {df['metrics/mAP50-95(B)'].max():.3f} (Epoch {df['metrics/mAP50-95(B)'].idxmax() + 1})")
        print(f"   Best Precision: {df['metrics/precision(B)'].max():.3f} (Epoch {df['metrics/precision(B)'].idxmax() + 1})")
        print(f"   Best Recall: {df['metrics/recall(B)'].max():.3f} (Epoch {df['metrics/recall(B)'].idxmax() + 1})")

### 6.1 Training Loss Curves

Visualize how training and validation losses evolved over epochs.

In [ ]:
if train_runs and results_csv.exists():
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot training losses
    axes[0, 0].plot(df.index + 1, df['train/box_loss'], label='Box Loss', linewidth=2)
    axes[0, 0].plot(df.index + 1, df['train/cls_loss'], label='Class Loss', linewidth=2)
    axes[0, 0].plot(df.index + 1, df['train/dfl_loss'], label='DFL Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Losses')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot validation losses
    axes[0, 1].plot(df.index + 1, df['val/box_loss'], label='Box Loss', linewidth=2, color='orange')
    axes[0, 1].plot(df.index + 1, df['val/cls_loss'], label='Class Loss', linewidth=2, color='red')
    axes[0, 1].plot(df.index + 1, df['val/dfl_loss'], label='DFL Loss', linewidth=2, color='purple')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].set_title('Validation Losses')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot precision and recall
    axes[1, 0].plot(df.index + 1, df['metrics/precision(B)'], label='Precision', linewidth=2, color='green')
    axes[1, 0].plot(df.index + 1, df['metrics/recall(B)'], label='Recall', linewidth=2, color='blue')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Score')
    axes[1, 0].set_title('Precision & Recall')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_ylim([0, 1])
    
    # Plot mAP scores
    axes[1, 1].plot(df.index + 1, df['metrics/mAP50(B)'], label='mAP@0.5', linewidth=2, color='darkgreen')
    axes[1, 1].plot(df.index + 1, df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', linewidth=2, color='darkblue')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('mAP')
    axes[1, 1].set_title('Mean Average Precision')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(output_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Training curves saved to {output_dir / 'training_curves.png'}")

## 7. Evaluate Model on Test Set

Load the best model and evaluate it on the test set (2,936 images).

In [ ]:
if train_runs:
    # Load the best model
    best_model_path = latest_run / 'weights' / 'best.pt'
    
    if not best_model_path.exists():
        print(f"❌ Best model not found: {best_model_path}")
    else:
        print(f"🔄 Loading best model from: {best_model_path}")
        best_model = YOLO(str(best_model_path))
        
        print(f"🧪 Evaluating on test set (2,936 images)...")
        print(f"   This will take approximately 5-10 minutes...\n")
        
        # Run evaluation on test set
        test_results = best_model.val(
            data=str(data_yaml_path),
            split='test',
            batch=16,
            imgsz=640,
            plots=True,
            save_json=True,
            project=str(project_root / 'runs' / 'detect'),
            name='test',
            exist_ok=True
        )
        
        print(f"\n✅ Test Set Evaluation Complete")
        print(f"\n📊 Test Set Metrics:")
        print(f"   Precision: {test_results.box.p[0]:.3f}")
        print(f"   Recall: {test_results.box.r[0]:.3f}")
        print(f"   mAP@0.5: {test_results.box.map50:.3f}")
        print(f"   mAP@0.5:0.95: {test_results.box.map:.3f}")
        
        # Check if meets requirements
        print(f"\n✅ Requirements Check:")
        req_precision = 0.90
        req_recall = 0.85
        req_map50 = 0.85
        
        precision_ok = test_results.box.p[0] >= req_precision
        recall_ok = test_results.box.r[0] >= req_recall
        map50_ok = test_results.box.map50 >= req_map50
        
        print(f"   Precision ≥ {req_precision}: {'✅' if precision_ok else '❌'} ({test_results.box.p[0]:.3f})")
        print(f"   Recall ≥ {req_recall}: {'✅' if recall_ok else '❌'} ({test_results.box.r[0]:.3f})")
        print(f"   mAP@0.5 ≥ {req_map50}: {'✅' if map50_ok else '❌'} ({test_results.box.map50:.3f})")
        
        if precision_ok and recall_ok and map50_ok:
            print(f"\n🎉 Model meets all performance requirements!")
        else:
            print(f"\n⚠️  Model does not meet all requirements. Consider:")
            print(f"   - Training for more epochs")
            print(f"   - Using a larger model (yolov8m or yolov8l)")
            print(f"   - Adjusting hyperparameters (learning rate, augmentation)")
else:
    print("❌ No training runs found. Please run the training cell first.")

## 8. Inference Examples

Run the trained model on sample test images and visualize predictions.

In [ ]:
if train_runs and best_model_path.exists():
    # Get random test images
    test_images_dir = dataset_root / data_config['test'] / 'images'
    test_images = list(test_images_dir.glob('*.jpg'))
    
    np.random.seed(42)
    sample_test_indices = np.random.choice(len(test_images), 4, replace=False)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    for idx, ax in enumerate(axes):
        img_path = test_images[sample_test_indices[idx]]
        
        # Run inference
        results = best_model.predict(
            source=str(img_path),
            conf=0.25,  # Confidence threshold
            imgsz=640,
            verbose=False
        )[0]
        
        # Get annotated image
        annotated_img = results.plot(
            line_width=2,
            font_size=12,
            labels=True,
            conf=True
        )
        
        # Convert BGR to RGB
        annotated_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
        
        # Count detections
        num_detections = len(results.boxes)
        
        ax.imshow(annotated_img)
        ax.set_title(f"{img_path.name}\n{num_detections} products detected")
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'inference_examples.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Inference examples saved to {output_dir / 'inference_examples.png'}")
else:
    print("❌ Best model not found. Please complete training first.")

## 9. Export Model

Copy the best model to the production models directory for use in the API.

In [ ]:
if train_runs and best_model_path.exists():
    # Create models directory
    models_dir = project_root / 'models'
    models_dir.mkdir(exist_ok=True)
    
    # Copy best model
    import shutil
    destination = models_dir / 'yolo_sku110k_best.pt'
    shutil.copy2(best_model_path, destination)
    
    # Get model file size
    model_size_mb = destination.stat().st_size / (1024 * 1024)
    
    print(f"✅ Model exported successfully!")
    print(f"   Source: {best_model_path}")
    print(f"   Destination: {destination}")
    print(f"   Size: {model_size_mb:.2f} MB")
    
    # Also save last checkpoint
    last_model_path = latest_run / 'weights' / 'last.pt'
    if last_model_path.exists():
        last_destination = models_dir / 'yolo_sku110k_last.pt'
        shutil.copy2(last_model_path, last_destination)
        print(f"\n   Last checkpoint also saved: {last_destination}")
    
    print(f"\n📋 Next Steps:")
    print(f"   1. Update config/settings.py to use: models/yolo_sku110k_best.pt")
    print(f"   2. Test ProductDetector with the new model")
    print(f"   3. Implement detect_products() and detect_gaps() methods")
    print(f"   4. Create API endpoints for image analysis")
else:
    print("❌ Best model not found. Please complete training first.")

## 10. Summary

### Training Configuration
- **Model**: YOLOv8s (small, 11M parameters)
- **Dataset**: SKU-110K (11,743 images, single "object" class)
- **Training**: 50 epochs, batch size 16, AdamW optimizer
- **Hardware**: Auto-detect (CPU/GPU)
- **Duration**: ~2-3 hours on M1/M2 Mac CPU

### Expected Results
After training, you should achieve:
- **Precision**: >90% (few false positives)
- **Recall**: >85% (catches most products)
- **mAP@0.5**: >85% (overall accuracy at 50% IoU threshold)
- **Inference Speed**: 300-400ms per image on M1/M2 Mac

### Files Generated
- `runs/detect/train/weights/best.pt` - Best model checkpoint
- `runs/detect/train/weights/last.pt` - Last epoch checkpoint
- `runs/detect/train/results.csv` - Training metrics
- `runs/detect/train/*.png` - Training plots
- `models/yolo_sku110k_best.pt` - Production model (exported)
- `outputs/training/*.png` - Notebook visualizations

### Next Steps (Phase 2 Remaining Tasks)
1. **T029**: ✅ Evaluate model (completed in this notebook)
2. **T030**: ✅ Export best checkpoint (completed in this notebook)
3. **T031**: Implement `detect_products()` method in ProductDetector
4. **T032**: Implement `detect_gaps()` algorithm for out-of-stock detection
5. **T033**: Create YOLO model wrapper in `models/yolo.py`
6. **T034-T035**: Implement API routers for analysis endpoints
7. **T036-T043**: Tests, notebooks, documentation

### Troubleshooting

**If training is slow:**
- Reduce batch size: `batch=8` or `batch=4`
- Use a smaller model: `yolov8n.pt` (nano, 3M params)
- Enable cache: Add `cache=True` to training config

**If metrics are low:**
- Train longer: Increase `epochs=100`
- Use larger model: `yolov8m.pt` or `yolov8l.pt`
- Adjust learning rate: Try `lr0=0.001` or `lr0=0.1`
- Check dataset quality: Verify annotations are correct

**If out of memory:**
- Reduce batch size: `batch=8` or `batch=4`
- Reduce image size: `imgsz=416` or `imgsz=320`
- Close other applications to free RAM

### References
- **YOLOv8 Docs**: https://docs.ultralytics.com
- **SKU-110K Paper**: https://arxiv.org/abs/1904.00853
- **Project Spec**: See `.specify/features/phase2.md`